# 26 — Capstone 2: Kaggle-Style EDA & Feature Engineering

End-to-end ML-ready pipeline on Titanic data: EDA → cleaning → feature engineering
→ target analysis → submission-ready DataFrame. Covers patterns for competitive ML.

In [ ]:
import pandas as pd
import numpy as np

# Load Titanic from stable URL
URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

try:
    df = pd.read_csv(URL)
    print(f"Loaded from URL: {df.shape}")
except Exception:
    # Fallback: generate synthetic Titanic-like data
    print("URL unavailable — generating synthetic Titanic data")
    np.random.seed(42)
    n = 891
    df = pd.DataFrame({
        'PassengerId': range(1, n+1),
        'Survived':    np.random.choice([0, 1], n, p=[0.617, 0.383]),
        'Pclass':      np.random.choice([1, 2, 3], n, p=[0.242, 0.207, 0.551]),
        'Name':        [f'Smith, Mr. John {i}' if np.random.random() > 0.5 else
                        f'Jones, Mrs. Alice {i}' for i in range(n)],
        'Sex':         np.random.choice(['male', 'female'], n, p=[0.647, 0.353]),
        'Age':         np.where(np.random.random(n) > 0.2,
                                np.random.uniform(1, 80, n).round(0), np.nan),
        'SibSp':       np.random.choice([0, 1, 2, 3, 4], n, p=[0.682, 0.234, 0.047, 0.023, 0.014]),
        'Parch':       np.random.choice([0, 1, 2, 3], n, p=[0.760, 0.134, 0.090, 0.016]),
        'Ticket':      [f'A/{np.random.randint(10000, 99999)}' for _ in range(n)],
        'Fare':        np.where(np.random.random(n) > 0.01,
                                np.random.exponential(30, n).round(4), np.nan),
        'Cabin':       np.where(np.random.random(n) > 0.7,
                                [f'{np.random.choice(list("ABCDEFG"))}{np.random.randint(1,150)}'
                                 for _ in range(n)], np.nan),
        'Embarked':    np.where(np.random.random(n) > 0.02,
                                np.random.choice(['S', 'C', 'Q'], n, p=[0.724, 0.188, 0.086]),
                                np.nan)
    })

df.head(3)

## 1. EDA — Exploratory Data Analysis

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print()
print(df.dtypes.to_string())

In [ ]:
# Missing value audit
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct':   (df.isna().mean() * 100).round(1),
    'dtype':         df.dtypes.astype(str)
})
print(missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False))

In [ ]:
# Survival rate by key features
print("Survival rate by Pclass:")
print(df.groupby('Pclass')['Survived'].mean().round(3))

print("\nSurvival rate by Sex:")
print(df.groupby('Sex')['Survived'].mean().round(3))

print("\nSurvival rate by Pclass × Sex:")
print(df.groupby(['Pclass', 'Sex'])['Survived'].mean().round(3).unstack())

In [ ]:
# Continuous feature distribution vs target
print("Age by survival:")
print(df.groupby('Survived')['Age'].describe().round(1))

print("\nFare by survival:")
print(df.groupby('Survived')['Fare'].describe().round(1))

## 2. Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Title extraction from Name
    df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.').str.strip()
    # Rare title consolidation
    rare = df['Title'].value_counts()
    rare_titles = rare[rare < 10].index.tolist()
    df['Title'] = df['Title'].replace(dict.fromkeys(rare_titles, 'Rare'))
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    # ── Family features
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
    df['FamilyGroup'] = pd.cut(
        df['FamilySize'],
        bins=[0, 1, 4, 20],
        labels=['Alone', 'Small', 'Large']
    )

    # ── Cabin features
    df['HasCabin']   = df['Cabin'].notna().astype(int)
    df['CabinDeck']  = df['Cabin'].str.extract(r'^([A-Z])').fillna('Unknown')

    # ── Fill missing Age (median by Title × Pclass)
    df['Age'] = df.groupby(['Title', 'Pclass'])['Age'].transform(
        lambda x: x.fillna(x.median())
    ).fillna(df['Age'].median())

    # ── Fill Fare and Embarked
    df['Fare']     = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # ── Age/Fare buckets
    df['AgeBand']  = pd.cut(df['Age'],  bins=[0, 12, 18, 35, 60, 100],
                             labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])
    df['FareBand'] = pd.qcut(df['Fare'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

    # ── Interaction features
    df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)  # female_1, male_3, etc.
    df['FarePerPerson'] = (df['Fare'] / df['FamilySize']).round(2)

    return df


df_feat = engineer_features(df)
print(f"After feature engineering: {df_feat.shape}")
print(df_feat[['Name', 'Title', 'Age', 'FamilySize', 'FamilyGroup', 'HasCabin', 'AgeBand']].head(5))

## 3. Feature vs Target Analysis

In [ ]:
# Survival rate for all engineered features
features_to_check = ['Title', 'FamilyGroup', 'AgeBand', 'FareBand', 'CabinDeck', 'Sex_Pclass']

for feat in features_to_check:
    sr = df_feat.groupby(feat, observed=True)['Survived'].agg(['mean', 'count'])
    sr.columns = ['survival_rate', 'count']
    sr['survival_rate'] = sr['survival_rate'].round(3)
    print(f"\nSurvival by {feat}:")
    print(sr.sort_values('survival_rate', ascending=False))

## 4. Encoding — ML-Ready Preparation

In [ ]:
# Select and encode features for ML
feature_cols = ['Pclass', 'Sex', 'Age', 'FamilySize', 'IsAlone', 'Fare',
                 'HasCabin', 'Title', 'FamilyGroup', 'AgeBand', 'FareBand',
                 'CabinDeck', 'Embarked', 'FarePerPerson']

X = df_feat[feature_cols].copy()
y = df_feat['Survived']

# One-hot encode categorical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical cols to encode: {cat_cols}")

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"\nX shape before encoding: {X.shape}")
print(f"X shape after encoding:  {X_encoded.shape}")
print(f"Feature names: {X_encoded.columns.tolist()}")

In [ ]:
# Verify no missing values remain
print(f"Missing in X: {X_encoded.isna().sum().sum()}")
print(f"Missing in y: {y.isna().sum()}")
print(f"\nTarget distribution:")
print(y.value_counts(normalize=True).round(3))

## 5. Correlation & Feature Importance (Pandas-only)

In [ ]:
# Point-biserial correlation with target (numeric features only)
numeric_feats = X_encoded.select_dtypes(include='number').columns.tolist()

correlations = (
    X_encoded[numeric_feats]
    .assign(Survived=y)
    .corr()['Survived']
    .drop('Survived')
    .abs()
    .sort_values(ascending=False)
)

print("Feature correlation with Survived (absolute):")
print(correlations.head(20).round(3).to_string())

## 6. Cross-Validation Ready Output

In [ ]:
# Final check — this is what you'd pass to sklearn
print("✅ X_encoded: ready for sklearn")
print(f"   Shape: {X_encoded.shape}")
print(f"   Dtypes: {X_encoded.dtypes.value_counts().to_dict()}")
print(f"   Missing: {X_encoded.isna().sum().sum()}")
print()
print("✅ y: target ready")
print(f"   Shape: {y.shape}")
print(f"   Values: {y.value_counts().to_dict()}")

## Key Kaggle Patterns Used

| Pattern | Code |
|---------|------|
| Title extraction | `str.extract(r', ([^.]+)\.')` |
| Rare-title grouping | `value_counts() + replace(dict.fromkeys(rare, 'Rare'))` |
| Group-aware Age fill | `groupby(['Title', 'Pclass']).transform(median fill)` |
| Cabin deck | `str.extract(r'^([A-Z])')` |
| Family size | `SibSp + Parch + 1` |
| Interaction feature | `Sex + '_' + Pclass.astype(str)` |
| One-hot encode | `pd.get_dummies(drop_first=True)` |
| Target correlation | `.corr()['target'].abs().sort_values()` |